## Comparison of early fusion (best model from Phase 1) and GNNs
Replace Morgan fingerprint encoder with GCN/GIN/GAT encoder\
Input: molecular graphs (instead of fingerprints)\
Output: 128-dim drug embedding (match fingerprint dimensionality)\
Keep everything downstream unchanged (fusion layer, genomics encoder, final prediction) 

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
molclr_path = os.path.abspath('../MolCLR')
if molclr_path not in sys.path:
    sys.path.append(molclr_path)
import pandas as pd
import numpy as np
import torch
import warnings
import gc
import torch.nn as nn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, GroupShuffleSplit, StratifiedKFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from torch_geometric.loader import DataLoader
from configs.config import HARMONIZED_DATA_PATH
from gpu.gnn_models_utils import add_gnn_embeddings_to_df, GCNDrugEncoder, GATDrugEncoder, GINDrugEncoder
from gpu.gnn_models_utils import MultimodalEarlyFusionNet, smiles_to_graph
from models.gcn_molclr import GCN
from torch.utils.data import DataLoader
from torch_geometric.data import Batch

In [ ]:
def early_fusion(df, target, model_name, drug_feature_type='morgan', apply_pca=True):
    """
    target: 'AUC' or 'IC50'
    model_name: 'RandomForest' or 'ElasticNet'
    drug_feature_type: 'morgan' (Phase 1 Baseline) or 'gnn' (Phase 2 GNN-Embeddings)
    apply_pca: True -> compresses the active ingredient features into 10 PCs (identical to Phase 1)
               False -> uses all GNN dimensions directly (scaled only)
    """
    df = df.copy()
    
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 
                      'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), index=df.index).astype('float32')
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)

    if drug_feature_type == 'morgan':
        if 'Bit_0' not in df.columns:
            fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index).astype('uint8')
            fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
            df = pd.concat([df, fp_df], axis=1)
        drug_cols = [col for col in df.columns if col.startswith('Bit_')]
    elif drug_feature_type == 'gnn':
        drug_cols = [col for col in df.columns if col.startswith('GNN_')]
        assert len(drug_cols) > 0, "No 'GNN_*' columns found in df! Run add_gnn_embeddings_to_df() first."
    else:
        raise ValueError("drug_feature_type must be 'morgan' or 'gnn'.")

    gene_cols = list(df.filter(regex=r'.* \(.*\)').columns)

    X = pd.concat([df[pharmacophores], df[drug_cols], df[gene_cols]], axis=1)
    y = df[target]

    '''
    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['DRUG_ID'].values))
    '''

    # Calculate the average IC50 value for each drug
    y_flat = np.array(y).flatten()
    graph_cache = {s: g for s in df['SMILES'].dropna().unique() if (g := smiles_to_graph(s)) is not None}
    df_sub = df[df['SMILES'].isin(graph_cache) & df['LN_IC50'].notna()].reset_index(drop=True)
    groups_flat = df_sub['SMILES'].values
    df_drugs = pd.DataFrame({'drug_group': groups_flat, 'LN_IC50': y_flat})
    drug_means = df_drugs.groupby('drug_group')['LN_IC50'].mean().reset_index() # combines all samples of the same medication and calculates the average target value for that specific medication.
    # pd.qcut to obtain 5 groups of medications of equal size
    drug_means['mean_bin'] = pd.qcut(drug_means['LN_IC50'], q=5, labels=False)
    # stratification only works with discrete values
    skf_test = StratifiedKFold(n_splits=10, shuffle=True, random_state=18092026) # divides the individual medications into groups based on their assigned class.
    train_val_drug_idx, test_drug_idx = next(skf_test.split(drug_means['drug_group'], drug_means['mean_bin']))
    test_drugs = drug_means.iloc[test_drug_idx]['drug_group'].values
    train_val_drugs = drug_means.iloc[train_val_drug_idx]['drug_group'].values
    # Create a new DataFrame containing only the remaining train/val drugs
    drug_means_train_val = drug_means.iloc[train_val_drug_idx].reset_index(drop=True)
    skf_val = StratifiedKFold(n_splits=9, shuffle=True, random_state=18092026)
    train_drug_idx, val_drug_idx = next(skf_val.split(drug_means_train_val['drug_group'], drug_means_train_val['mean_bin']))
    val_drugs = drug_means_train_val.iloc[val_drug_idx]['drug_group'].values
    train_drugs = drug_means_train_val.iloc[train_drug_idx]['drug_group'].values
    # Map Back to the Sample Level
    test_idx = np.where(np.isin(groups_flat, test_drugs))[0]
    val_idx = np.where(np.isin(groups_flat, val_drugs))[0]
    train_idx = np.where(np.isin(groups_flat, train_drugs))[0]
    print(f"Samples - Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups = df.iloc[train_idx]['DRUG_ID'].values

    # Preprocessing
    if apply_pca:
        drug_transformer = (
            Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=10, random_state=42))])
            if drug_feature_type == 'gnn'
            else PCA(n_components=10, random_state=42)
        )
    else:
        drug_transformer = StandardScaler()

    preprocessor = ColumnTransformer(
        transformers=[
            ('pharmacophore', StandardScaler(), pharmacophores),
            ('drug', drug_transformer, drug_cols)
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )
    preprocessor.set_output(transform="pandas")

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features='sqrt',
            n_jobs=4,
            random_state=42
        )

        print(f"Starting Cross-Validation on Training Data ({drug_feature_type.upper()})...")
        cv_results = cross_validate(
            model, X_train_transformed, y_train,
            groups=groups,
            cv=GroupKFold(5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        rmse_scores = np.sqrt(-cv_results['test_neg_mean_squared_error'])
        r2_scores = cv_results['test_r2']

        print(f"--- Multimodal (Random Forest - Early Fusion | {drug_feature_type.upper()}) ---")
        print(f"CV R² Score:   {np.mean(r2_scores):.4f}")
        print(f"CV RMSE:       {np.mean(rmse_scores):.4f}")

        model.fit(X_train_transformed, y_train)
        y_pred = model.predict(X_test_transformed)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        print("-" * 45)

        all_features = X_train_transformed.columns
        importances = model.feature_importances_
        gene_set = set(gene_cols)

        features_df = pd.DataFrame({
            'Feature_Name': all_features,
            'Importance': importances,
            'Type': ['Genetic' if f in gene_set else 'Molecular structure' for f in all_features]
        })

        selected_features = features_df[features_df['Importance'] > 0.0001]
        print(f"Selected features: {len(selected_features)} / {len(all_features)} "
              f"(Chem: {(selected_features['Type'] == 'Molecular structure').sum()} | "
              f"Bio: {(selected_features['Type'] == 'Genetic').sum()})")
        print("\nTop 10 most important features:")
        print(features_df.sort_values(by='Importance', ascending=False).head(10))

        return {'cv_r2': np.mean(r2_scores), 'cv_rmse': np.mean(rmse_scores),
                'test_r2': test_r2, 'test_rmse': test_rmse}

    elif model_name == 'ElasticNet':
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1],
                cv=list(GroupKFold(n_splits=5).split(X_train_transformed, y_train, groups=groups)),
                max_iter=8000,
                alphas=20,
                tol=1e-3,
                random_state=42,
                n_jobs=1
            ))
        ])

        pipeline.fit(X_train_transformed, y_train)
        fitted_model = pipeline.named_steps['model']

        y_pred = pipeline.predict(X_test_transformed)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        best_mean_mse = np.min(np.mean(fitted_model.mse_path_, axis=-1))
        train_cv_r2 = 1 - (best_mean_mse / np.var(y_train))

        print(f"\n--- Elastic Net (Early Fusion | {drug_feature_type.upper()}) ---")
        print(f"Chosen Alpha:    {fitted_model.alpha_:.6f} | L1-Ratio: {fitted_model.l1_ratio_:.2f}")
        print(f"CV R² Score:     {train_cv_r2:.4f}")
        print(f"Test R² Score:   {test_r2:.4f}")
        print(f"Test RMSE:       {test_rmse:.4f}")
        print("-" * 45)

        feature_names = X_train_transformed.columns
        features_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': fitted_model.coef_})
        selected_features = features_df[features_df['Coefficient'] != 0]

        print(f"Elastic Net selected {len(selected_features)} / {len(feature_names)} features.")
        print(f"Top 5 Positive:\n{features_df.sort_values(by='Coefficient', ascending=False).head(5)}")
        print(f"Top 5 Negative:\n{features_df.sort_values(by='Coefficient', ascending=True).head(5)}")

        return {'cv_r2': train_cv_r2, 'test_r2': test_r2, 'test_rmse': test_rmse}

In [14]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
df = pd.read_parquet(HARMONIZED_DATA_PATH)
# get pretrained weights of MolCLR
molclr_gcn = GCN(num_layer=5, emb_dim=300, feat_dim=512, drop_ratio=0, pool="mean").to(device)
molclr_dir = Path("../MolCLR").resolve()
checkpoint_path = molclr_dir / "ckpt" / "pretrained_gcn" / "checkpoints" / "model.pth"
state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
molclr_gcn.load_state_dict(state_dict, strict=True)

encoders = {
    "GCN (128-dim)": GCNDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "GIN (128-dim)": GINDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "GAT (128-dim)": GATDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "MolCLR (Pre-trained)": molclr_gcn
}

### Random Forest

In [15]:
results_rf = {}

# Phase-1-Baseline (Morgan FP)
results_rf["Morgan FP (Baseline)"] = early_fusion(df, target='LN_IC50', model_name='RandomForest', drug_feature_type='morgan')

# All GNN-Encoder
for name, encoder in encoders.items():
    print(f"\n================ {name} ================")
    df_gnn = add_gnn_embeddings_to_df(df, encoder, device=device)
    results_rf[name] = early_fusion(df_gnn, target='LN_IC50', model_name='RandomForest', drug_feature_type='gnn', apply_pca=True)

pd.DataFrame(results_rf).T

Samples - Train: 116293 | Val: 14548 | Test: 13241
Starting Cross-Validation on Training Data (MORGAN)...
--- Multimodal (Random Forest - Early Fusion | MORGAN) ---
CV R² Score:   0.0300
CV RMSE:       2.7358
Test R² Score: 0.1092
Test RMSE:     2.5832
---------------------------------------------
Selected features: 984 / 996 (Chem: 18 | Bio: 966)

Top 10 most important features:
   Feature_Name  Importance                 Type
12         pca4    0.061528  Molecular structure
15         pca7    0.057727  Molecular structure
17         pca9    0.053191  Molecular structure
14         pca6    0.046683  Molecular structure
10         pca2    0.046004  Molecular structure
8          pca0    0.045003  Molecular structure
11         pca3    0.044242  Molecular structure
13         pca5    0.041283  Molecular structure
3    Hydrophobe    0.041062  Molecular structure
16         pca8    0.040882  Molecular structure

================ GCN (128-dim) ================
Embeddings created: 128 dimen

,cv_r2,cv_rmse,test_r2,test_rmse
Morgan FP (Baseline),0.029979,2.735847,0.109204,2.583185
GCN (128-dim),0.177220,2.516317,0.186354,2.468789
GIN (128-dim),0.164383,2.537884,0.270894,2.337016
GAT (128-dim),0.204995,2.472562,0.122846,2.563328
MolCLR (Pre-trained),0.171975,2.520024,-0.027045,2.773710


-> MolCLR (and also GAT) shows strong overfitting!!

### Elastic Net

In [16]:
results_en = {}

# Phase-1-Baseline (Morgan FP)
results_en["Morgan FP (Baseline)"] = early_fusion(df, target='LN_IC50', model_name='ElasticNet', drug_feature_type='morgan')

# All GNN-Encoder
for name, encoder in encoders.items():
    print(f"\n================ {name} ================")
    df_gnn = add_gnn_embeddings_to_df(df, encoder, device=device)
    results_en[name] = early_fusion(df_gnn, target='LN_IC50', model_name='ElasticNet', drug_feature_type='gnn', apply_pca=True)

pd.DataFrame(results_en).T

Samples - Train: 116293 | Val: 14548 | Test: 13241

--- Elastic Net (Early Fusion | MORGAN) ---
Chosen Alpha:    0.201602 | L1-Ratio: 0.05
CV R² Score:     0.0798
Test R² Score:   0.1170
Test RMSE:       2.5718
---------------------------------------------
Elastic Net selected 352 / 996 features.
Top 5 Positive:
       Feature  Coefficient
10        pca2     0.332296
3   Hydrophobe     0.312869
14        pca6     0.206295
12        pca4     0.206257
15        pca7     0.123412
Top 5 Negative:
             Feature  Coefficient
6       NegIonizable    -0.349503
4   LumpedHydrophobe    -0.199166
1           Acceptor    -0.166862
11              pca3    -0.116305
16              pca8    -0.104884

================ GCN (128-dim) ================
Embeddings created: 128 dimensions for 229 unique active ingredients.
Samples - Train: 116293 | Val: 14548 | Test: 13241

--- Elastic Net (Early Fusion | GNN) ---
Chosen Alpha:    0.580944 | L1-Ratio: 0.05
CV R² Score:     0.1299
Test R² Score:   0.

,cv_r2,test_r2,test_rmse
Morgan FP (Baseline),0.079848,0.117031,2.571811
GCN (128-dim),0.129941,0.066612,2.644220
GIN (128-dim),0.196275,0.105167,2.589031
GAT (128-dim),0.141497,0.043037,2.677404
MolCLR (Pre-trained),0.200253,0.133188,2.548172


### Next step: Train GNNs 

We wanna train the GNNs but in combination with Elastic Net/Random Forest, it is not possible.
So we build our own model for training and then use these weights for early fusion above

#### GCN & GIN

In [17]:
graph_cache = {s: g for s in df['SMILES'].dropna().unique() if (g := smiles_to_graph(s)) is not None}
df_sub = df[df['SMILES'].isin(graph_cache) & df['LN_IC50'].notna()].reset_index(drop=True)

gene_cols = list(df_sub.filter(regex=r'.* \(.*\)').columns)
X_tab = df_sub[gene_cols].values.astype(np.float32)
y = df_sub['LN_IC50'].values.astype(np.float32)
graphs = [graph_cache[s] for s in df_sub['SMILES']]

collate = lambda b: (Batch.from_data_list([x[0] for x in b]), torch.stack([x[1] for x in b]), torch.stack([x[2] for x in b]))
get_loader = lambda idx, X_s, shuf=False: DataLoader(
    [(graphs[i], torch.tensor(X_s[k]), torch.tensor(y[i])) for k, i in enumerate(idx)],
    batch_size=64, shuffle=shuf, collate_fn=collate, drop_last=shuf
)

# Training + Held-out drug evaluation
os.makedirs("checkpoints", exist_ok=True)
'''
drug_groups = df_sub['DRUG_ID' if 'DRUG_ID' in df_sub.columns else 'SMILES'].values
tr_all, te_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X_tab, y, drug_groups))
tr_sub, va_sub = next(GroupShuffleSplit(1, test_size=0.15, random_state=42).split(X_tab[tr_all], y[tr_all], drug_groups[tr_all]))
tr_idx, va_idx = tr_all[tr_sub], tr_all[va_sub]
'''
# Train/Validation split for held-out drug evaluation
# Calculate the average IC50 value for each drug
y_flat = np.array(y).flatten()
groups_flat = df_sub['SMILES'].values
df_drugs = pd.DataFrame({'drug_group': groups_flat, 'LN_IC50': y_flat})
drug_means = df_drugs.groupby('drug_group')['LN_IC50'].mean().reset_index() # combines all samples of the same medication and calculates the average target value for that specific medication.
# pd.qcut to obtain 5 groups of medications of equal size
drug_means['mean_bin'] = pd.qcut(drug_means['LN_IC50'], q=5, labels=False)
# stratification only works with discrete values
skf_test = StratifiedKFold(n_splits=10, shuffle=True, random_state=18092026) # divides the individual medications into groups based on their assigned class.
train_val_drug_idx, test_drug_idx = next(skf_test.split(drug_means['drug_group'], drug_means['mean_bin']))
test_drugs = drug_means.iloc[test_drug_idx]['drug_group'].values
train_val_drugs = drug_means.iloc[train_val_drug_idx]['drug_group'].values
# Create a new DataFrame containing only the remaining train/val drugs
drug_means_train_val = drug_means.iloc[train_val_drug_idx].reset_index(drop=True)
skf_val = StratifiedKFold(n_splits=9, shuffle=True, random_state=18092026)
train_drug_idx, val_drug_idx = next(skf_val.split(drug_means_train_val['drug_group'], drug_means_train_val['mean_bin']))
val_drugs = drug_means_train_val.iloc[val_drug_idx]['drug_group'].values
train_drugs = drug_means_train_val.iloc[train_drug_idx]['drug_group'].values
# Map Back to the Sample Level
te_idx = np.where(np.isin(groups_flat, test_drugs))[0]
va_idx = np.where(np.isin(groups_flat, val_drugs))[0]
tr_idx = np.where(np.isin(groups_flat, train_drugs))[0]
print(f"Samples - Train: {len(tr_idx)} | Val: {len(va_idx)} | Test: {len(te_idx)}")

def train_held_out_drugs(name, encoder_fn, lr=1e-3, epochs=80, patience=10):
    torch.manual_seed(42)
    scaler = StandardScaler()
    tr_loader = get_loader(tr_idx, scaler.fit_transform(X_tab[tr_idx]), shuf=True)
    va_loader = get_loader(va_idx, scaler.transform(X_tab[va_idx]))
    te_loader = get_loader(te_idx, scaler.transform(X_tab[te_idx]))

    model = MultimodalEarlyFusionNet(encoder_fn(), genomics_dim=X_tab.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    best_loss, wait, best_w = float('inf'), 0, None
    for _ in range(epochs):
        model.train()
        for g, x, target in tr_loader:
            opt.zero_grad()
            loss = loss_fn(model(g.to(device), x.to(device)), target.to(device))
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_preds = torch.cat([model(g.to(device), x.to(device)) for g, x, _ in va_loader]).cpu().numpy()
        val_loss = mean_squared_error(y[va_idx], val_preds)

        if val_loss < best_loss - 1e-4:
            best_loss, wait, best_w = val_loss, 0, copy.deepcopy(model.state_dict())
        elif (wait := wait + 1) >= patience:
            break

    model.load_state_dict(best_w)
    torch.save(model.drug_encoder.state_dict(), f"checkpoints/{name.lower()}_trained.pt")

    model.eval()
    with torch.no_grad():
        test_preds = torch.cat([model(g.to(device), x.to(device)) for g, x, _ in te_loader]).cpu().numpy()

    return {
        "Held-out Drug R²": round(r2_score(y[te_idx], test_preds), 4),
        "Held-out Drug RMSE": round(np.sqrt(mean_squared_error(y[te_idx], test_preds)), 4),
    }

node_dim, edge_dim = graphs[0].x.shape[1], graphs[0].edge_attr.shape[1]
results = pd.DataFrame({
    "GCN": train_held_out_drugs("GCN", lambda: GCNDrugEncoder(node_dim, hidden_dim=128)),
    "GIN": train_held_out_drugs("GIN", lambda: GINDrugEncoder(node_dim, hidden_dim=128)),
}).T
display(results)

Samples - Train: 116293 | Val: 14548 | Test: 13241


,Held-out Drug R²,Held-out Drug RMSE
GCN,0.2180,2.4203
GIN,0.0834,2.6203


### Next: evaluate Early fusion with pre-trained weights of GNNs

In [18]:
sample_g = smiles_to_graph(df['SMILES'].dropna().iloc[0])
node_dim = sample_g.x.shape[1]

trained_encoders = {
    "GCN (Trained)": (GCNDrugEncoder(node_dim, hidden_dim=128), "checkpoints/gcn_trained.pt"),
    "GIN (Trained)": (GINDrugEncoder(node_dim, hidden_dim=128), "checkpoints/gin_trained.pt"),
}

results_rf = {}
results_en = {}

for name, (encoder, ckpt_path) in trained_encoders.items():
    # load weights
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
    encoder.load_state_dict(state_dict)
    encoder = encoder.to(device).eval()
    # extract embeddings
    df_gnn = add_gnn_embeddings_to_df(df, encoder, device=device)
    # Random Forest & ElasticNet
    results_rf[name] = early_fusion(
        df_gnn, target='LN_IC50', model_name='RandomForest', drug_feature_type='gnn', apply_pca=True
    )
    results_en[name] = early_fusion(
        df_gnn, target='LN_IC50', model_name='ElasticNet', drug_feature_type='gnn', apply_pca=True
    )

print("=== Random Forest (Pre-trained GNN-Embeddings) ===")
display(pd.DataFrame(results_rf).T)

print("=== ElasticNet (Pre-trained GNN-Embeddings) ===")
display(pd.DataFrame(results_en).T)

Embeddings created: 128 dimensions for 229 unique active ingredients.
Samples - Train: 116293 | Val: 14548 | Test: 13241
Starting Cross-Validation on Training Data (GNN)...
--- Multimodal (Random Forest - Early Fusion | GNN) ---
CV R² Score:   0.2885
CV RMSE:       2.3365
Test R² Score: 0.2463
Test RMSE:     2.3762
---------------------------------------------
Selected features: 915 / 996 (Chem: 18 | Bio: 897)

Top 10 most important features:
   Feature_Name  Importance                 Type
8          pca0    0.129259  Molecular structure
11         pca3    0.105519  Molecular structure
13         pca5    0.066578  Molecular structure
9          pca1    0.064658  Molecular structure
14         pca6    0.045802  Molecular structure
10         pca2    0.042489  Molecular structure
12         pca4    0.040921  Molecular structure
16         pca8    0.039062  Molecular structure
17         pca9    0.037936  Molecular structure
15         pca7    0.037012  Molecular structure
Samples - Trai

,cv_r2,cv_rmse,test_r2,test_rmse
GCN (Trained),0.288483,2.336538,0.246254,2.376178
GIN (Trained),0.407454,2.130390,0.095142,2.603494


=== ElasticNet (Pre-trained GNN-Embeddings) ===


,cv_r2,test_r2,test_rmse
GCN (Trained),0.372702,-0.025975,2.772264
GIN (Trained),0.620734,-0.127569,2.906284
